# BlitzMart Analytics: Data Generation & Cleaning

This notebook builds the foundation of the project. A synthetic but realistic dataset for **BlitzMart**, a fictional German e-commerce retailer modeled on companies like Zalando, Otto, and Amazon DE.

Three stages here:

1. **Data generation.** Create 8 relational tables (~6.6M rows) representing customers, orders, products, shipments, returns, plus supporting reference data.
2. **Quality injection.** Add realistic data problems (missing values, duplicates, outliers, invalid records) so the cleaning stage actually has something to do.
3. **Cleaning pipeline.** Detect and fix every issue, then write an audit log.

The output is a clean, analysis-ready dataset stored in Google Drive. The SQL and Python notebooks use this data.

## 1. Setup

### 1.1 Mount Drive

Project lives in Drive so the generated CSVs persist between Colab sessions.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### 1.2 Project folders

Set up the folder structure. `exist_ok=True` so this cell can be re-run safely.

In [ ]:
import os

PROJECT_PATH = "/content/drive/MyDrive/BlitzMart_Analytics"
DATA_PATH = f"{PROJECT_PATH}/data"

os.makedirs(DATA_PATH, exist_ok=True)
print(f"Project folder ready: {PROJECT_PATH}")
print(f"Data folder ready:    {DATA_PATH}")

### 1.3 Install Faker

`Faker` generates realistic fake names. The `de_DE` locale (set below) produces German names, which keeps the dataset feeling authentic.

In [ ]:
pip install faker -q

## 2. Data Generation

Build all 8 tables in one cell. Reference tables first (warehouses, suppliers, products), then everything that depends on them (customers, orders, order_items, shipments, returns).

A few design choices that matter for the downstream analysis:

- **Seasonality.** Nov/Dec get 3x weight, Jun/Jul 1.5x. Creates the Q4 holiday spike.
- **Basket sizes.** Most orders have 1-2 items, long tail up to 6. Matches typical e-commerce.
- **Pricing.** Selling price = cost * (1.3 to 2.5). Realistic margin spread.
- **Status mix.** ~78% delivered, ~10% returned, ~5% shipped, ~4% cancelled, ~3% processing.
- **Reproducible.** Seed 42 everywhere. Re-running this cell gives identical data.

| Table | Rows | What it has |
|---|---|---|
| warehouses | 6 | Fulfilment centres across Germany |
| suppliers | 50 | Supplier list with lead times |
| products | 5,000 | Catalogue across 8 categories |
| customers | 200,000 | Customers across 15 cities |
| orders | 1,500,000 | Order headers |
| order_items | ~3.4M | Line items (biggest table) |
| shipments | ~1.4M | Delivery records |
| returns | ~150K | Returned orders |

In [ ]:
import os
import random
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
from faker import Faker

# seed for reproducibility
np.random.seed(42)
random.seed(42)
fake = Faker("de_DE")
Faker.seed(42)

OUTPUT_DIR = "/content/drive/MyDrive/BlitzMart_Analytics/data"
os.makedirs(OUTPUT_DIR, exist_ok=True)

N_CUSTOMERS = 200_000
N_PRODUCTS = 5_000
N_SUPPLIERS = 50
N_WAREHOUSES = 6
N_ORDERS = 1_500_000

START_DATE = datetime(2022, 1, 1)
END_DATE = datetime(2024, 12, 31)
SIGNUP_START = datetime(2020, 1, 1)

print(f"Generating ~6.5M rows across 8 tables.")
print(f"Output: {OUTPUT_DIR}\n")

# 1. WAREHOUSES (6 hubs, 4 regions)
print("1/8 warehouses...")
warehouses_df = pd.DataFrame([
    {"warehouse_id": 1, "warehouse_name": "Berlin Hub",            "city": "Berlin",            "postal_code": "10115", "region": "East",  "capacity_units": 500000},
    {"warehouse_id": 2, "warehouse_name": "Hamburg DC",            "city": "Hamburg",           "postal_code": "20095", "region": "North", "capacity_units": 450000},
    {"warehouse_id": 3, "warehouse_name": "Munich Center",         "city": "München",           "postal_code": "80331", "region": "South", "capacity_units": 600000},
    {"warehouse_id": 4, "warehouse_name": "Frankfurt Logistics",   "city": "Frankfurt am Main", "postal_code": "60311", "region": "West",  "capacity_units": 550000},
    {"warehouse_id": 5, "warehouse_name": "Köln Fulfillment",      "city": "Köln",              "postal_code": "50667", "region": "West",  "capacity_units": 400000},
    {"warehouse_id": 6, "warehouse_name": "Leipzig Distribution",  "city": "Leipzig",           "postal_code": "04109", "region": "East",  "capacity_units": 350000},
])
warehouses_df.to_csv(f"{OUTPUT_DIR}/warehouses.csv", index=False)

# 2. SUPPLIERS (mostly Germany, some EU and Asia)
print("2/8 suppliers...")
countries = ["Germany"]*4 + ["Netherlands", "Poland", "Czech Republic", "France", "Italy", "China", "Vietnam", "Turkey"]
suppliers_df = pd.DataFrame([{
    "supplier_id": i,
    "supplier_name": fake.company(),
    "country": random.choice(countries),
    "rating": round(np.random.uniform(3.0, 5.0), 2),
    "lead_time_days": int(np.random.randint(2, 45)),
} for i in range(1, N_SUPPLIERS + 1)])
suppliers_df.to_csv(f"{OUTPUT_DIR}/suppliers.csv", index=False)

# 3. PRODUCTS
# each category gets its own cost range. selling price = cost * 1.3 to 2.5
print("3/8 products...")
categories = {
    "Electronics":       (50, 1500),
    "Fashion":           (15, 250),
    "Home & Kitchen":    (10, 400),
    "Beauty":            (5,  100),
    "Sports & Outdoors": (20, 600),
    "Books":             (5,  50),
    "Toys":              (8,  150),
    "Automotive":        (15, 800),
}
cat_list = list(categories.keys())
products_rows = []
for i in range(1, N_PRODUCTS + 1):
    cat = random.choice(cat_list)
    min_p, max_p = categories[cat]
    cost = round(np.random.uniform(min_p, max_p), 2)
    price = round(cost * np.random.uniform(1.3, 2.5), 2)
    products_rows.append({
        "product_id": i,
        "product_name": f"{fake.word().capitalize()}-{cat[:3].upper()}-{i}",
        "category": cat,
        "supplier_id": int(np.random.randint(1, N_SUPPLIERS + 1)),
        "unit_cost": cost,
        "unit_price": price,
        "weight_kg": round(np.random.uniform(0.1, 15.0), 2),
    })
products_df = pd.DataFrame(products_rows)
products_df.to_csv(f"{OUTPUT_DIR}/products.csv", index=False)

# 4. CUSTOMERS
# names drawn from a pool of 2000 first/last names. faster than calling Faker 200k times
print(f"4/8 customers ({N_CUSTOMERS:,})...")
german_cities = ["Berlin", "Hamburg", "München", "Köln", "Frankfurt am Main",
                 "Stuttgart", "Düsseldorf", "Leipzig", "Dortmund", "Essen",
                 "Bremen", "Dresden", "Hannover", "Nürnberg", "Duisburg"]
city_regions = {
    "Berlin": "East", "Hamburg": "North", "München": "South", "Köln": "West",
    "Frankfurt am Main": "West", "Stuttgart": "South", "Düsseldorf": "West",
    "Leipzig": "East", "Dortmund": "West", "Essen": "West", "Bremen": "North",
    "Dresden": "East", "Hannover": "North", "Nürnberg": "South", "Duisburg": "West",
}

first_name_pool = [fake.first_name() for _ in range(2000)]
last_name_pool = [fake.last_name() for _ in range(2000)]

customer_ids = np.arange(1, N_CUSTOMERS + 1)
cities = np.random.choice(german_cities, size=N_CUSTOMERS)
regions = np.array([city_regions[c] for c in cities])
signup_days = (END_DATE - SIGNUP_START).days
signup_offsets = np.random.randint(0, signup_days, size=N_CUSTOMERS)
signup_dates = [SIGNUP_START + timedelta(days=int(d)) for d in signup_offsets]
ages = np.random.randint(18, 75, size=N_CUSTOMERS)
genders = np.random.choice(["M", "F", "D"], size=N_CUSTOMERS, p=[0.48, 0.49, 0.03])
segments = np.random.choice(["Standard", "Premium", "VIP"], size=N_CUSTOMERS, p=[0.75, 0.20, 0.05])

customers_df = pd.DataFrame({
    "customer_id": customer_ids,
    "first_name": np.random.choice(first_name_pool, size=N_CUSTOMERS),
    "last_name": np.random.choice(last_name_pool, size=N_CUSTOMERS),
    "city": cities,
    "region": regions,
    "age": ages,
    "gender": genders,
    "signup_date": signup_dates,
    "customer_segment": segments,
})
customers_df.to_csv(f"{OUTPUT_DIR}/customers.csv", index=False)

# 5. ORDERS
# date weights: Nov/Dec 3x, Jun/Jul 1.5x, all other months 1x
# this is what creates the holiday spike pattern
print(f"5/8 orders ({N_ORDERS:,})...")
order_ids = np.arange(1, N_ORDERS + 1)
customer_assignments = np.random.choice(customer_ids, size=N_ORDERS)

days_range = (END_DATE - START_DATE).days + 1
day_weights = np.ones(days_range)
for d in range(days_range):
    m = (START_DATE + timedelta(days=d)).month
    if m in (11, 12):
        day_weights[d] = 3.0
    elif m in (6, 7):
        day_weights[d] = 1.5
day_weights /= day_weights.sum()
date_offsets = np.random.choice(days_range, size=N_ORDERS, p=day_weights)
order_dates = pd.to_datetime([START_DATE + timedelta(days=int(d)) for d in date_offsets])

warehouse_assignments = np.random.randint(1, N_WAREHOUSES + 1, size=N_ORDERS)
statuses = np.random.choice(
    ["Delivered", "Shipped", "Cancelled", "Processing", "Returned"],
    size=N_ORDERS, p=[0.78, 0.05, 0.04, 0.03, 0.10],
)
payments = np.random.choice(
    ["Credit Card", "PayPal", "Klarna", "SEPA Direct Debit", "Apple Pay", "Google Pay"],
    size=N_ORDERS, p=[0.30, 0.25, 0.20, 0.15, 0.05, 0.05],
)

orders_df = pd.DataFrame({
    "order_id": order_ids,
    "customer_id": customer_assignments,
    "order_date": order_dates,
    "warehouse_id": warehouse_assignments,
    "order_status": statuses,
    "payment_method": payments,
})
orders_df.to_csv(f"{OUTPUT_DIR}/orders.csv", index=False)

# 6. ORDER_ITEMS (biggest table, takes about a minute)
# np.repeat trick: expand order_ids by their item count, then vectorize the rest
print("6/8 order_items (biggest table, takes a minute)...")
items_per_order = np.random.choice(
    [1, 2, 3, 4, 5, 6], size=N_ORDERS,
    p=[0.35, 0.30, 0.18, 0.10, 0.05, 0.02],
)
total_items = int(items_per_order.sum())
print(f"     total order_items rows: {total_items:,}")

order_id_repeated = np.repeat(order_ids, items_per_order)
product_assignments = np.random.choice(np.arange(1, N_PRODUCTS + 1), size=total_items)
quantities = np.random.choice([1, 2, 3, 4], size=total_items, p=[0.65, 0.20, 0.10, 0.05])

# vectorized price lookup, much faster than .merge
price_lookup = products_df.set_index("product_id")["unit_price"].to_numpy()
unit_prices = price_lookup[product_assignments - 1]
discounts = np.random.choice([0.0, 0.05, 0.10, 0.15, 0.20],
                             size=total_items, p=[0.60, 0.15, 0.15, 0.07, 0.03])
line_totals = np.round(unit_prices * quantities * (1 - discounts), 2)

order_items_df = pd.DataFrame({
    "order_item_id": np.arange(1, total_items + 1),
    "order_id": order_id_repeated,
    "product_id": product_assignments,
    "quantity": quantities,
    "unit_price": unit_prices,
    "discount": discounts,
    "line_total": line_totals,
})
order_items_df.to_csv(f"{OUTPUT_DIR}/order_items.csv", index=False)

# 7. SHIPMENTS (one per delivered/shipped/returned order)
print("7/8 shipments...")
shipped_mask = orders_df["order_status"].isin(["Delivered", "Shipped", "Returned"]).to_numpy()
shipped_order_ids = orders_df.loc[shipped_mask, "order_id"].to_numpy()
shipped_order_dates = orders_df.loc[shipped_mask, "order_date"].to_numpy()
n_shipments = len(shipped_order_ids)
print(f"     total shipments: {n_shipments:,}")

carriers = np.random.choice(["DHL", "Hermes", "DPD", "UPS", "GLS"],
                            size=n_shipments, p=[0.50, 0.20, 0.15, 0.08, 0.07])
promised_days = np.random.choice([1, 2, 3, 4, 5],
                                 size=n_shipments, p=[0.05, 0.20, 0.50, 0.20, 0.05])
delivery_jitter = np.random.choice([-1, 0, 1, 2, 3, 5],
                                   size=n_shipments, p=[0.05, 0.55, 0.20, 0.10, 0.07, 0.03])
actual_days = np.clip(promised_days + delivery_jitter, 1, None)

ship_dates = pd.to_datetime(shipped_order_dates) + pd.to_timedelta(
    np.random.randint(0, 2, size=n_shipments), unit="D")
delivery_dates = ship_dates + pd.to_timedelta(actual_days, unit="D")
on_time = (actual_days <= promised_days).astype(int)

shipments_df = pd.DataFrame({
    "shipment_id": np.arange(1, n_shipments + 1),
    "order_id": shipped_order_ids,
    "carrier": carriers,
    "ship_date": ship_dates,
    "promised_delivery_days": promised_days,
    "actual_delivery_days": actual_days,
    "delivery_date": delivery_dates,
    "on_time_delivery": on_time,
})
shipments_df.to_csv(f"{OUTPUT_DIR}/shipments.csv", index=False)

# 8. RETURNS (one per returned order)
print("8/8 returns...")
returned_mask = (orders_df["order_status"] == "Returned").to_numpy()
returned_order_ids = orders_df.loc[returned_mask, "order_id"].to_numpy()
returned_order_dates = orders_df.loc[returned_mask, "order_date"].to_numpy()
n_returns = len(returned_order_ids)
print(f"     total returns: {n_returns:,}")

reasons = np.random.choice(
    ["Wrong size", "Damaged in transit", "Not as described",
     "Changed mind", "Quality issue", "Wrong item shipped"],
    size=n_returns, p=[0.30, 0.15, 0.15, 0.20, 0.15, 0.05],
)
return_offsets = np.random.randint(3, 30, size=n_returns)
return_dates = pd.to_datetime(returned_order_dates) + pd.to_timedelta(return_offsets, unit="D")
refunds = np.round(np.random.uniform(20, 500, size=n_returns), 2)

returns_df = pd.DataFrame({
    "return_id": np.arange(1, n_returns + 1),
    "order_id": returned_order_ids,
    "return_date": return_dates,
    "return_reason": reasons,
    "refund_amount": refunds,
})
returns_df.to_csv(f"{OUTPUT_DIR}/returns.csv", index=False)

# sanity check
print("\n=== DONE ===")
total_rows = 0
for f in sorted(os.listdir(OUTPUT_DIR)):
    path = os.path.join(OUTPUT_DIR, f)
    size_mb = os.path.getsize(path) / (1024 * 1024)
    rows = sum(1 for _ in open(path, encoding="utf-8")) - 1
    total_rows += rows
    print(f"  {f:<22}  {rows:>10,} rows   {size_mb:>7.1f} MB")
print(f"\nTOTAL ROWS: {total_rows:,}")

## 3. Data Quality Injection

Real data is never clean. So before cleaning anything, I deliberately inject realistic problems into the dataset. This way the cleaning pipeline has actual work to do, and the results are defensible.

The 6 issue types I inject:

| Issue | Mimics | Volume |
|---|---|---|
| Missing customer ages | Optional field skipped at signup | 2% of customers |
| Missing genders | Same | 1% of customers |
| Inconsistent city casing | Free text entry on signup form | 3% of customers |
| Duplicate orders | Payment retry or system glitch | 0.1% of orders |
| Invalid quantities, zero prices | Refund processing bugs | ~0.5% of items |
| Impossible delivery dates | Timezone bug, delivered before shipped | 0.3% of shipments |
| Extreme refund outliers | Missing decimal in data entry | 50 records |
| Missing supplier references | Data migration left orphans | 1% of products |

Dirty data goes into a separate `data_dirty/` folder. The clean data stays untouched.

In [ ]:
import os
import numpy as np
import pandas as pd

np.random.seed(42)

DATA_PATH = "/content/drive/MyDrive/BlitzMart_Analytics/data"
DIRTY_PATH = "/content/drive/MyDrive/BlitzMart_Analytics/data_dirty"
os.makedirs(DIRTY_PATH, exist_ok=True)

print("Loading clean data...")
customers   = pd.read_csv(f"{DATA_PATH}/customers.csv")
orders      = pd.read_csv(f"{DATA_PATH}/orders.csv")
order_items = pd.read_csv(f"{DATA_PATH}/order_items.csv")
products    = pd.read_csv(f"{DATA_PATH}/products.csv")
shipments   = pd.read_csv(f"{DATA_PATH}/shipments.csv")
returns     = pd.read_csv(f"{DATA_PATH}/returns.csv")

print("Injecting realistic data quality issues...\n")

# 1. CUSTOMERS: nulls and messy city names

# 2% null ages (skipped signup field)
null_age_idx = np.random.choice(customers.index, size=int(len(customers) * 0.02), replace=False)
customers.loc[null_age_idx, "age"] = np.nan

# 3% city names made messy. simulates free-text "berlin", "BERLIN", " Berlin " kind of inputs
messy_idx = np.random.choice(customers.index, size=int(len(customers) * 0.03), replace=False)
messy_transforms = [str.lower, str.upper, lambda s: f"  {s}  ", lambda s: f"{s} "]
for i in messy_idx:
    transform = np.random.choice(messy_transforms)
    customers.loc[i, "city"] = transform(customers.loc[i, "city"])

# 1% null gender
null_gender_idx = np.random.choice(customers.index, size=int(len(customers) * 0.01), replace=False)
customers.loc[null_gender_idx, "gender"] = np.nan

print(f"Customers: injected {len(null_age_idx):,} null ages, {len(messy_idx):,} messy cities, {len(null_gender_idx):,} null genders")

# 2. ORDERS: duplicates from payment retries
dup_count = int(len(orders) * 0.001)
duplicate_rows = orders.sample(n=dup_count, random_state=42)
orders_dirty = pd.concat([orders, duplicate_rows], ignore_index=True)
print(f"Orders: injected {dup_count:,} duplicate rows")

# 3. ORDER_ITEMS: invalid qty and zero price
invalid_qty_idx = np.random.choice(order_items.index, size=int(len(order_items) * 0.003), replace=False)
order_items.loc[invalid_qty_idx, "quantity"] = np.random.choice([0, -1, -2], size=len(invalid_qty_idx))

zero_price_idx = np.random.choice(order_items.index, size=int(len(order_items) * 0.002), replace=False)
order_items.loc[zero_price_idx, "unit_price"] = 0.0

print(f"Order items: injected {len(invalid_qty_idx):,} invalid quantities, {len(zero_price_idx):,} zero prices")

# 4. SHIPMENTS: impossible dates (delivery before ship). timezone-bug style
bad_date_idx = np.random.choice(shipments.index, size=int(len(shipments) * 0.003), replace=False)
shipments["ship_date"] = pd.to_datetime(shipments["ship_date"])
shipments["delivery_date"] = pd.to_datetime(shipments["delivery_date"])
shipments.loc[bad_date_idx, "delivery_date"] = shipments.loc[bad_date_idx, "ship_date"] - pd.Timedelta(days=1)
print(f"Shipments: injected {len(bad_date_idx):,} impossible delivery dates")

# 5. RETURNS: outlier refunds (data entry error, missing decimal place)
outlier_idx = np.random.choice(returns.index, size=50, replace=False)
returns.loc[outlier_idx, "refund_amount"] = np.random.uniform(50000, 99999, size=50)
print(f"Returns: injected 50 outlier refund amounts")

# 6. PRODUCTS: orphan supplier refs
null_supplier_idx = np.random.choice(products.index, size=int(len(products) * 0.01), replace=False)
products.loc[null_supplier_idx, "supplier_id"] = np.nan
print(f"Products: injected {len(null_supplier_idx):,} null supplier_ids")

# save dirty versions
customers.to_csv(f"{DIRTY_PATH}/customers.csv", index=False)
orders_dirty.to_csv(f"{DIRTY_PATH}/orders.csv", index=False)
order_items.to_csv(f"{DIRTY_PATH}/order_items.csv", index=False)
products.to_csv(f"{DIRTY_PATH}/products.csv", index=False)
shipments.to_csv(f"{DIRTY_PATH}/shipments.csv", index=False)
returns.to_csv(f"{DIRTY_PATH}/returns.csv", index=False)

# warehouses + suppliers had nothing injected, copy as-is so all 8 tables are in dirty folder
import shutil
shutil.copy(f"{DATA_PATH}/warehouses.csv", f"{DIRTY_PATH}/warehouses.csv")
shutil.copy(f"{DATA_PATH}/suppliers.csv",  f"{DIRTY_PATH}/suppliers.csv")

print("\n=== DIRTY DATA SAVED ===")
print(f"Location: {DIRTY_PATH}")

## 4. Cleaning Pipeline & Audit

The actual ETL step. Load dirty data, detect each issue, apply the fix, log the action. Every action goes into an audit CSV so the work is traceable.

A few decisions I made per issue type:

- **Missing ages.** Median imputation. Keeps the record, preserves distribution. Dropping the row would lose otherwise-valid customer info.
- **Messy city names.** Strip whitespace, title case. "berlin", "BERLIN", " Berlin " all collapse back to "Berlin".
- **Missing genders.** Explicit "Unknown" label. Don't try to guess. Analysis-safe.
- **Duplicate orders.** Keep first occurrence, drop rest. Standard dedup.
- **Invalid quantities and zero prices.** Drop the rows. A negative quantity or free product has no valid business meaning.
- **Impossible delivery dates.** Drop. Delivery before ship is physically impossible.
- **Outlier refunds.** IQR flag, then cap at 5,000 EUR. Don't delete (the return still happened).
- **Missing supplier refs.** Assign to a placeholder "Unknown Supplier" with ID 0. Add that supplier row so FK joins still resolve.

In [ ]:
import os
import numpy as np
import pandas as pd

DIRTY_PATH   = "/content/drive/MyDrive/BlitzMart_Analytics/data_dirty"
CLEAN_PATH   = "/content/drive/MyDrive/BlitzMart_Analytics/data_clean"
os.makedirs(CLEAN_PATH, exist_ok=True)

# every cleaning action gets logged here for the audit CSV
audit_log = []

def log(table, issue, count, action):
    audit_log.append({"table": table, "issue": issue, "rows_affected": count, "action": action})
    print(f"  [{table}] {issue}: {count:,} rows -> {action}")

print("="*70)
print("DATA QUALITY AUDIT & CLEANING PIPELINE")
print("="*70)

print("\nLoading dirty data...")
customers   = pd.read_csv(f"{DIRTY_PATH}/customers.csv")
orders      = pd.read_csv(f"{DIRTY_PATH}/orders.csv")
order_items = pd.read_csv(f"{DIRTY_PATH}/order_items.csv")
products    = pd.read_csv(f"{DIRTY_PATH}/products.csv")
shipments   = pd.read_csv(f"{DIRTY_PATH}/shipments.csv", parse_dates=["ship_date", "delivery_date"])
returns     = pd.read_csv(f"{DIRTY_PATH}/returns.csv", parse_dates=["return_date"])
warehouses  = pd.read_csv(f"{DIRTY_PATH}/warehouses.csv")
suppliers   = pd.read_csv(f"{DIRTY_PATH}/suppliers.csv")

# ===== CUSTOMERS =====
print("\n[1/6] Cleaning CUSTOMERS...")

# missing ages -> median imputation
null_ages = customers["age"].isna().sum()
median_age = customers["age"].median()
customers["age"] = customers["age"].fillna(median_age).astype(int)
log("customers", "missing age", null_ages, f"filled with median ({median_age:.0f})")

# messy city casing -> strip + title case to collapse variants
before_unique = customers["city"].nunique()
customers["city"] = customers["city"].str.strip().str.title()
customers["city"] = customers["city"].replace({"München": "München", "Köln": "Köln"})
after_unique = customers["city"].nunique()
log("customers", "inconsistent city formatting", before_unique - after_unique, "standardized casing & whitespace")

# null gender -> "Unknown"
null_gender = customers["gender"].isna().sum()
customers["gender"] = customers["gender"].fillna("Unknown")
log("customers", "missing gender", null_gender, "filled with 'Unknown'")

# ===== ORDERS =====
print("\n[2/6] Cleaning ORDERS...")

# duplicate order_ids -> keep first
before = len(orders)
orders = orders.drop_duplicates(subset=["order_id"], keep="first")
dup_removed = before - len(orders)
log("orders", "duplicate order_ids", dup_removed, "removed (kept first occurrence)")

# ===== ORDER_ITEMS =====
print("\n[3/6] Cleaning ORDER_ITEMS...")

# invalid quantity (0 or negative) -> drop
before = len(order_items)
order_items = order_items[order_items["quantity"] > 0].copy()
qty_removed = before - len(order_items)
log("order_items", "invalid quantity (<=0)", qty_removed, "rows removed")

# zero price -> drop
before = len(order_items)
order_items = order_items[order_items["unit_price"] > 0].copy()
price_removed = before - len(order_items)
log("order_items", "zero unit_price", price_removed, "rows removed")

# recalculate line_total to stay consistent with cleaned data
order_items["line_total"] = (
    order_items["quantity"] * order_items["unit_price"] * (1 - order_items["discount"])
).round(2)
log("order_items", "line_total recalculated", len(order_items), "ensured consistency")

# ===== SHIPMENTS =====
print("\n[4/6] Cleaning SHIPMENTS...")

# delivery before ship -> physically impossible, drop
bad_dates = (shipments["delivery_date"] < shipments["ship_date"]).sum()
shipments = shipments[shipments["delivery_date"] >= shipments["ship_date"]].copy()
log("shipments", "delivery before ship date", bad_dates, "rows removed")

# ===== RETURNS =====
print("\n[5/6] Cleaning RETURNS...")

# outlier refunds -> IQR flag, cap at 5,000 EUR (don't delete, return still happened)
q1 = returns["refund_amount"].quantile(0.25)
q3 = returns["refund_amount"].quantile(0.75)
iqr = q3 - q1
upper_bound = q3 + 1.5 * iqr
outliers = (returns["refund_amount"] > upper_bound).sum()
returns.loc[returns["refund_amount"] > 5000, "refund_amount"] = 5000.0
log("returns", f"outlier refunds (>EUR {upper_bound:.0f})", outliers, "capped at EUR 5,000")

# ===== PRODUCTS =====
print("\n[6/6] Cleaning PRODUCTS...")

# missing supplier_id -> assign placeholder (id=0)
# also add Unknown Supplier row so FK joins resolve cleanly
null_supp = products["supplier_id"].isna().sum()
products["supplier_id"] = products["supplier_id"].fillna(0).astype(int)

if 0 not in suppliers["supplier_id"].values:
    unknown_supp = pd.DataFrame([{
        "supplier_id": 0, "supplier_name": "Unknown Supplier",
        "country": "Unknown", "rating": 0.0, "lead_time_days": 0
    }])
    suppliers = pd.concat([unknown_supp, suppliers], ignore_index=True)

log("products", "missing supplier_id", null_supp, "assigned to 'Unknown Supplier' (id=0)")

# save clean data
print("\n" + "="*70)
print("SAVING CLEAN DATA...")
customers.to_csv(f"{CLEAN_PATH}/customers.csv", index=False)
orders.to_csv(f"{CLEAN_PATH}/orders.csv", index=False)
order_items.to_csv(f"{CLEAN_PATH}/order_items.csv", index=False)
products.to_csv(f"{CLEAN_PATH}/products.csv", index=False)
shipments.to_csv(f"{CLEAN_PATH}/shipments.csv", index=False)
returns.to_csv(f"{CLEAN_PATH}/returns.csv", index=False)
warehouses.to_csv(f"{CLEAN_PATH}/warehouses.csv", index=False)
suppliers.to_csv(f"{CLEAN_PATH}/suppliers.csv", index=False)

# write audit log
audit_df = pd.DataFrame(audit_log)
audit_df.to_csv(f"{CLEAN_PATH}/_cleaning_report.csv", index=False)

total_fixed = audit_df["rows_affected"].sum()

print("\n" + "="*70)
print("DATA QUALITY REPORT")
print("="*70)
print(audit_df.to_string(index=False))
print("\n" + "-"*70)
print(f"TOTAL ROWS AFFECTED: {total_fixed:,}")
print(f"Clean data saved to: {CLEAN_PATH}")
print(f"Audit report saved:  {CLEAN_PATH}/_cleaning_report.csv")
print("="*70)

## 5. Audit Summary: Fixes vs Validation Checks

The raw cleaning report counts every row the pipeline touched. But not every touched row was actually a problem. Recalculating `line_total` for consistency, for example, runs across millions of rows, but those rows weren't broken. Just a routine integrity check.

It would be misleading to claim "3.4 million quality issues fixed" when the real fix count is much smaller. So this cell splits the two:

- **Data fixes.** Actual problems corrected (missing values, dupes, invalid records, outliers).
- **Validation checks.** Routine integrity ops run across the data.

Both numbers get exported separately. The fix count is the defensible headline number.

In [ ]:
import pandas as pd

CLEAN_PATH = "/content/drive/MyDrive/BlitzMart_Analytics/data_clean"

audit = pd.read_csv(f"{CLEAN_PATH}/_cleaning_report.csv")

# tag each action as fix vs validation based on keywords
validation_keywords = ["recalculated", "ensured consistency"]
audit["category"] = audit["action"].apply(
    lambda x: "Validation" if any(k in x.lower() for k in validation_keywords) else "Data Fix"
)

fixes       = audit[audit["category"] == "Data Fix"].copy()
validations = audit[audit["category"] == "Validation"].copy()

total_fixes     = fixes["rows_affected"].sum()
total_validated = validations["rows_affected"].sum()

print("="*72)
print("DATA QUALITY SUMMARY")
print("="*72)

print("\nACTUAL DATA QUALITY ISSUES FIXED")
print("-"*72)
print(fixes[["table", "issue", "rows_affected", "action"]].to_string(index=False))
print(f"\n   Total rows fixed: {total_fixes:,}")

print("\nVALIDATION CHECKS PERFORMED")
print("-"*72)
print(validations[["table", "issue", "rows_affected", "action"]].to_string(index=False))
print(f"\n   Total rows validated: {total_validated:,}")

print("\n" + "="*72)
print("HEADLINE METRICS")
print("="*72)
print(f"   Total records processed:          6,641,392")
print(f"   Data quality issues fixed:        {total_fixes:,}")
print(f"   Data integrity checks run:        {total_validated:,}")
print(f"   Pipeline tables cleaned:          8")
print(f"   Cleaning techniques applied:      6 (imputation, deduplication,")
print(f"                                        standardization, outlier capping,")
print(f"                                        type coercion, referential repair)")
print("="*72)

# export the two splits
fixes.to_csv(f"{CLEAN_PATH}/_data_fixes.csv", index=False)
validations.to_csv(f"{CLEAN_PATH}/_validations.csv", index=False)
print(f"\nFiles saved:")
print(f"  {CLEAN_PATH}/_data_fixes.csv")
print(f"  {CLEAN_PATH}/_validations.csv")

---

## Output

After running everything, the project's Drive folder has three data folders:

| Folder | What |
|---|---|
| `data/` | Original generated data, clean by construction |
| `data_dirty/` | Same data with realistic quality issues injected |
| `data_clean/` | Fully cleaned, analysis-ready + audit logs |

`data_clean/` is the source of truth for Notebook 2 (SQL) and Notebook 3 (Python).